In [1]:
import pandas as pd
df = pd.read_csv(r'C:\DATA ANALYSIS AND DATA SCIENCE PROJECTS\Accident Severity Prediction\data\clean data\final_dataset.csv') 

In [2]:
def map_fatality_level(x):
    if x == 1:
        return 'Low Fatality'
    else:
        return 'High Fatality'

df['Fatality_Level'] = df['Fatalities_Count'].apply(map_fatality_level)
df['Fatality_Level_Binary'] = df['Fatality_Level'].map({'Low Fatality': 0, 'High Fatality': 1})

In [3]:
df["Restraint_Used"] = df["Killed_Restraint_List"].astype(str).str.contains(
    "Lapbelt|Shoulder_Lapbelt", case=False, na=False).astype(int)

features = df.copy()
drop_cols = [
    "Crash_ID", "State_Name", "State_Code", "City_Name",
    "Notif_Hour", "Notif_Minute", 
    "Arrival_Hour", "Arrival_Minute",
    "Hospital_Hour", "Hospital_Minute",
    "Fatalities_Count", "Total_Killed", "Total_Alive",
    "Fatal_Seats_List", "Killed_Restraint_List",
    "Fatality_Level_Binary", "Latitude", "Longitude"
]
features = features.drop(columns=drop_cols, errors='ignore')

numeric_features = [
    'Count_Vehicles_Total', 'Count_Occupants_MV', 'Count_Non_Motorists',
    'Crash_Hour', 'Crash_Minute',
    'Max_Driver_Age', 'Min_Driver_Age',
    'Total_Exceeding_Limit_Vehicles', 'Total_Too_Fast_Vehicles',
    'Driver_Max_Prev_Accident_Count', 'Driver_Max_Prev_Speeding_Count'
]

categorical_features = [
    'Clean_Light_Condition', 'Clean_Weather_Condition',
    'Area_Rural_Urban', 'Road_Functional_Class',
    'Intersection_Type', 'Driver_License_Category',
    'Light_Condition', 'Weather_Condition',
    'Crash_Road_Surface_Condition', 'Crash_Horizontal_Alignment',
    'Crash_Vertical_Profile', 'Crash_Impairment_Category',
    'Crash_Vehicle_Defect', 'On_NHS_Road',
    'Any_Driver_Drinking', 'Driver_Sex_List'
]

In [4]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

target = 'Fatality_Level_Binary'
feature_cols = numeric_features + categorical_features + ['Restraint_Used']

X = features[feature_cols]
y = df[target]
for col in categorical_features:
    X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

C:\Users\Hp\AppData\Local\Temp\ipykernel_4532\708323677.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype('category')
C:\Users\Hp\AppData\Local\Temp\ipykernel_4532\708323677.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype('category')
C:\Users\Hp\AppData\Local\Temp\ipykernel_4532\708323677.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [5]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

base_lgbm = LGBMClassifier(
    objective='binary',
    class_weight='balanced',
    random_state=42
)

param_distributions = {
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 5, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 300, 500, 800],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.0, 0.1, 0.5, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=base_lgbm,
    param_distributions=param_distributions,
    n_iter=40, 
    scoring='roc_auc',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

best_lgbm = random_search.best_estimator_

y_pred = best_lgbm.predict(X_test)
y_prob = best_lgbm.predict_proba(X_test)[:, 1]

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 2108, number of negative: 28015
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003621 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 479
[LightGBM] [Info] Number of data points in the train set: 30123, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [6]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_param_dist = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

X_train_rf = pd.get_dummies(X_train, drop_first=True)
X_test_rf = pd.get_dummies(X_test, drop_first=True)
X_train_rf, X_test_rf = X_train_rf.align(X_test_rf, join='left', axis=1, fill_value=0)

rf_base = RandomForestClassifier(
    class_weight='balanced_subsample', 
    random_state=42
    )

rf_random = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

rf_random.fit(X_train_rf, y_train)
best_rf = rf_random.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [7]:
import re
def clean_col_names(df):
    new_cols = []
    for col in df.columns:
        cleaned_col = re.sub(r'[\[\]<]', '_', col) 
        new_cols.append(re.sub(r'[^A-Za-z0-9_]+', '', cleaned_col))
    df.columns = new_cols
    return df

X_train_xgb = clean_col_names(X_train_rf.copy())
X_test_xgb = clean_col_names(X_test_rf.copy())
X_train_xgb, X_test_xgb = X_train_xgb.align(
X_test_xgb, join='left', axis=1, fill_value=0
)

In [8]:
from xgboost import XGBClassifier

xgb_param_dist = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'min_child_weight': [1, 3, 5]
}

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=30,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

xgb_random.fit(X_train_xgb, y_train)
best_xgb = xgb_random.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


c:\Users\Hp\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:12:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

X_train_final = X_train_xgb.copy()
X_test_final = X_test_xgb.copy()

stack_model = StackingClassifier(
    estimators=[
        ('lgbm', best_lgbm),  
        ('rf', best_rf),
        ('xgb', best_xgb)
    ],
    final_estimator = LogisticRegression(max_iter=1000, class_weight='balanced'),
    cv = 3,
    n_jobs = -1,
    passthrough = False
)

stack_model.fit(X_train_final, y_train)
y_pred_stack = stack_model.predict(X_test_final)
y_prob_stack = stack_model.predict_proba(X_test_final)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("ROC AUC:", roc_auc_score(y_test, y_prob_stack))
print("\nClassification Report:\n", classification_report(y_test, y_pred_stack))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_stack))

Accuracy: 0.7348293719293586
ROC AUC: 0.8878621270361094

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.72      0.83      7004
           1       0.20      0.91      0.33       527

    accuracy                           0.73      7531
   macro avg       0.59      0.82      0.58      7531
weighted avg       0.94      0.73      0.80      7531


Confusion Matrix:
 [[5053 1951]
 [  46  481]]
